## Ejercicio 2: Escalamiento de tickets de soporte técnico

Diseño de un agente inteligente que asigna cada ticket al nivel de soporte adecuado a partir del tiempo de espera, la urgencia reportada y la condición premium del cliente.

### Ficha PEAS

| Componente | Descripción |
|---|---|
| **P — Medida de desempeño** | Reducir el tiempo de resolución y los incumplimientos de prioridad, evitando escalaciones innecesarias y manteniendo la satisfacción del cliente. |
| **E — Entorno** | Mesa de ayuda, cola de tickets, clientes con distintos contratos y equipo de soporte organizado en niveles de atención. |
| **A — Acciones** | Mantener en nivel 1, escalar a nivel 2 o escalar a nivel 3. |
| **S — Percepciones** | Tiempo de espera en minutos, nivel de urgencia (`baja`, `media` o `alta`) y condición de cliente premium. |

**Objetivo:** dirigir cada ticket al nivel capaz de atenderlo oportunamente, priorizando los incidentes críticos y los compromisos de servicio sin sobrecargar innecesariamente a los niveles superiores.

### Justificación de las reglas

Se aplicarán las siguientes reglas de decisión:

- Los tickets de urgencia **alta** se escalan inmediatamente al nivel 3.
- Los tickets de urgencia **media** se escalan al nivel 2 cuando acumulan 30 minutos de espera o pertenecen a un cliente premium; en los demás casos permanecen en nivel 1.
- Los tickets de urgencia **baja** se escalan al nivel 2 desde los 45 minutos si el cliente es premium, o desde los 90 minutos si no lo es; en los demás casos permanecen en nivel 1.

La urgencia alta requiere especialistas de inmediato porque una demora puede aumentar el impacto del incidente.
En urgencia media, el tiempo acumulado o el compromiso premium justifican adelantar la atención sin ocupar el nivel 3.
Los casos de baja urgencia permanecen inicialmente en el nivel 1 para evitar sobrecargar al equipo especializado.
Los límites de espera impiden que un ticket quede estancado y permiten priorizar antes a los clientes premium.

### Código

In [ ]:
def agente_soporte(tiempo_espera_minutos, nivel_urgencia, cliente_premium):
    """Decide el nivel de soporte y retorna una tupla (acción, motivo)."""
    if isinstance(tiempo_espera_minutos, bool) or not isinstance(tiempo_espera_minutos, int):
        raise TypeError("El tiempo de espera debe ser un número entero.")
    if tiempo_espera_minutos < 0:
        raise ValueError("El tiempo de espera no puede ser negativo.")
    if not isinstance(nivel_urgencia, str):
        raise TypeError("El nivel de urgencia debe ser texto.")
    if not isinstance(cliente_premium, bool):
        raise TypeError("La condición premium debe ser True o False.")

    urgencia = nivel_urgencia.strip().lower()
    if urgencia not in {"baja", "media", "alta"}:
        raise ValueError("La urgencia debe ser 'baja', 'media' o 'alta'.")

    if urgencia == "alta":
        return (
            "escalar a nivel 3",
            "urgencia alta: requiere atención especializada inmediata",
        )

    if urgencia == "media":
        if tiempo_espera_minutos >= 30:
            return (
                "escalar a nivel 2",
                f"urgencia media con {tiempo_espera_minutos} minutos de espera",
            )
        if cliente_premium:
            return (
                "escalar a nivel 2",
                "urgencia media de un cliente premium",
            )
        return ("mantener en nivel 1", "urgencia media con espera menor a 30 minutos")

    limite_espera = 45 if cliente_premium else 90
    if tiempo_espera_minutos >= limite_espera:
        tipo_cliente = "premium" if cliente_premium else "no premium"
        return (
            "escalar a nivel 2",
            f"urgencia baja, cliente {tipo_cliente} y {tiempo_espera_minutos} minutos de espera",
        )

    return (
        "mantener en nivel 1",
        f"urgencia baja y espera menor al límite de {limite_espera} minutos",
    )

### Simulación y pruebas

Se prueban nueve tickets que cubren las tres acciones, clientes premium y no premium, casos con condiciones que no coinciden entre sí y los límites exactos de espera.

In [ ]:
casos_prueba = [
    # espera, urgencia, premium, acción esperada, descripción
    (0, "alta", False, "escalar a nivel 3", "alta urgencia sin espera"),
    (120, "alta", True, "escalar a nivel 3", "alta urgencia, premium y espera alta"),
    (10, "media", False, "mantener en nivel 1", "urgencia media con espera corta"),
    (30, "media", False, "escalar a nivel 2", "límite de espera para urgencia media"),
    (5, "media", True, "escalar a nivel 2", "premium con urgencia media y espera corta"),
    (44, "baja", True, "mantener en nivel 1", "premium justo antes de su límite"),
    (45, "baja", True, "escalar a nivel 2", "urgencia baja, premium y espera alta"),
    (89, "baja", False, "mantener en nivel 1", "no premium justo antes de su límite"),
    (90, "baja", False, "escalar a nivel 2", "límite para urgencia baja no premium"),
]

for numero, (espera, urgencia, premium, esperado, descripcion) in enumerate(casos_prueba, start=1):
    accion, motivo = agente_soporte(espera, urgencia, premium)
    assert accion == esperado, f"Caso {numero}: se esperaba '{esperado}' y se obtuvo '{accion}'"
    print(f"Caso {numero}: {descripcion}")
    print(f"  Entrada: espera={espera} min, urgencia={urgencia}, premium={premium}")
    print(f"  Decisión: {accion}")
    print(f"  Motivo: {motivo}\n")